# Análise de Resultados dos Experimentos

In [154]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Configurações globais de exibição do Pandas
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", "{:.6f}".format)

### Leitura dos Dados e Pré-processamento

In [155]:
csv_path = "out/all_results.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"O arquivo {csv_path} não foi encontrado. Execute os experimentos primeiro.")

df = pd.read_csv(csv_path)
if "variation" not in df.columns:
    raise ValueError(
        "O CSV está no schema antigo (sem coluna 'variation'). Re-execute o "
        "experimentos.ipynb com o pipeline novo (apague a pasta out/ antes)."
    )
print(f"Foram carregados {len(df)} registros do arquivo CSV.")

# Mapeamento do algoritmo base (sem sufixo de variação).
name_map = {"MOEAD": "MOEA/D", "NSGAII": "NSGA-II", "NSGAIII": "NSGA-III"}
df["base_algorithm"] = df["algorithm"].map(lambda a: name_map.get(a, a))

# Mapeamento e ordem canônica das variações de orçamento.
VAR_DISPLAY = {
    "pure_moea": "MOEA Puro",
    "dvl_10": "DVL 10%",
    "dvl_25": "DVL 25%",
    "dvl_50": "DVL 50%",
    "dvl_75": "DVL 75%",
}
VAR_ORDER = ["pure_moea", "dvl_10", "dvl_25", "dvl_50", "dvl_75"]
df["display_variation"] = df["variation"].map(VAR_DISPLAY)

Foram carregados 11520 registros do arquivo CSV.


### Geração das Tabelas Resumidas por Algoritmo

In [156]:
# Função personalizada para converter o DataFrame em Markdown (sem depender da biblioteca tabulate)
def to_markdown_custom(df_table):
    headers = df_table.columns.tolist()
    header_row = "| " + " | ".join(map(str, headers)) + " |"
    separator_row = "| " + " | ".join(["---"] * len(headers)) + " |"
    data_rows = []
    for idx, row in df_table.iterrows():
        data_rows.append("| " + " | ".join(map(lambda x: str(x).replace("\n", " "), row.values)) + " |")
    return "\n".join([header_row, separator_row] + data_rows)

def generate_summary_table(algo_df):
    if algo_df.empty:
        return pd.DataFrame()
    grouped = algo_df.groupby(["problem", "m", "max_evaluations", "base_algorithm", "variation"])
    
    rows = []
    for (problem, m, max_eval, base_algo, variation), group in grouped:
        # Hypervolume
        hv_mean = group["hypervolume"].mean()
        hv_std = group["hypervolume"].std()
        
        # Tempo de Execução (Total)
        time_mean = group["cpu_time_seconds"].mean()
        time_std = group["cpu_time_seconds"].std()
        
        # Tempo de Treinamento (DVL)
        train_time_mean = group["model_training_time"].mean() if "model_training_time" in group.columns else np.nan
        train_time_std = group["model_training_time"].std() if "model_training_time" in group.columns else np.nan
        
        # Tempo de Avaliação Real
        eval_time_mean = group["real_evaluation_time"].mean() if "real_evaluation_time" in group.columns else np.nan
        eval_time_std = group["real_evaluation_time"].std() if "real_evaluation_time" in group.columns else np.nan
        
        # Chamadas de F.O. (Avaliações Reais Totais)
        evals_mean = group["objective_calls"].mean() if "objective_calls" in group.columns else np.nan
        evals_std = group["objective_calls"].std() if "objective_calls" in group.columns else np.nan
        
        # Avaliações consumidas no DVL e no MOEA
        dvl_evals_mean = group["dvl_evaluations"].mean() if "dvl_evaluations" in group.columns else 0.0
        dvl_evals_std = group["dvl_evaluations"].std() if "dvl_evaluations" in group.columns else 0.0
        
        moea_evals_mean = group["moea_evaluations"].mean() if "moea_evaluations" in group.columns else 0.0
        moea_evals_std = group["moea_evaluations"].std() if "moea_evaluations" in group.columns else 0.0
        
        # Sucesso
        success_count = (group["status"] == "success").sum()
        total_count = len(group)
        
        # Formatação das strings
        def fmt_val(mean, std):
            if pd.isna(mean):
                return "N/A"
            if pd.isna(std):
                std = 0.0
            return f"{mean:.6f} ± {std:.6f}"
            
        def fmt_int(mean, std):
            if pd.isna(mean):
                return "N/A"
            if pd.isna(std):
                std = 0.0
            return f"{mean:.1f} ± {std:.1f}"
            
        def fmt_int_or_float(mean, std):
            if pd.isna(mean):
                return "N/A"
            if pd.isna(std) or std == 0.0:
                return f"{int(round(mean))}"
            return f"{mean:.1f} ± {std:.1f}"

        hv_str = fmt_val(hv_mean, hv_std)
        time_str = fmt_val(time_mean, time_std)
        train_time_str = fmt_val(train_time_mean, train_time_std)
        eval_time_str = fmt_val(eval_time_mean, eval_time_std)
        evals_str = fmt_int(evals_mean, evals_std)
        success_str = f"{success_count}/{total_count}"
        
        # Nome amigável do algoritmo / variação
        if variation == "pure_moea":
            display_name = base_algo
        else:
            display_name = f"{base_algo} + {VAR_DISPLAY.get(variation, variation)}"
        
        rows.append({
            "Problema": problem,
            "Objetivos": m,
            "Avaliações": max_eval,
            "Algoritmo": display_name,
            "variation_raw": variation,
            "Hypervolume": hv_str,
            "Chamadas F.O.": evals_str,
            "Avaliações DVL": fmt_int_or_float(dvl_evals_mean, dvl_evals_std),
            "Avaliações MOEA": fmt_int_or_float(moea_evals_mean, moea_evals_std),
            "Treino Modelos (s)": train_time_str,
            "Avaliação F.O. (s)": eval_time_str,
            "Tempo Total (s)": time_str,
            "Sucesso": success_str
        })
        
    summary_table = pd.DataFrame(rows)
    if not summary_table.empty:
        summary_table["variation_raw"] = pd.Categorical(
            summary_table["variation_raw"], categories=VAR_ORDER, ordered=True
        )
        summary_table = summary_table.sort_values(by=["Problema", "Objetivos", "Avaliações", "variation_raw"]).drop(columns=["variation_raw"])
    return summary_table

base_algorithms = ["MOEA/D", "NSGA-II", "NSGA-III"]

for base_algo in base_algorithms:
    # Exibe o título principal do Algoritmo
    display(HTML(f"<h3 style='color: #1a5f7a; margin-top: 40px; font-weight: bold; border-bottom: 2px solid #1a5f7a; padding-bottom: 5px;'>Resultados: {base_algo}</h3>"))
    
    # Filtrar dados para o algoritmo base (incluindo todas as variações)
    df_algo = df[df["base_algorithm"] == base_algo]
    table_summary = generate_summary_table(df_algo)
    
    # Estilização CSS com cores de texto explícitas
    table_style = """
    <style>
        .custom-table-wrapper {
            width: 100%;
            border: 1px solid #ddd;
            border-radius: 8px;
            padding: 15px;
            background-color: #ffffff !important;
            color: #222222 !important;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
            margin-top: 15px;
            margin-bottom: 30px;
        }
        .custom-table-wrapper table {
            width: 100% !important;
            border-collapse: collapse;
            margin: 0 !important;
            color: #222222 !important;
        }
        .custom-table-wrapper th {
            background-color: #1a5f7a !important;
            color: white !important;
            font-weight: bold;
            padding: 8px !important;
            text-align: left !important;
        }
        .custom-table-wrapper td {
            padding: 6px 8px !important;
            border-bottom: 1px solid #eee !important;
            text-align: left !important;
            color: #222222 !important;
        }
        .custom-table-wrapper tr:hover {
            background-color: #f5f5f5 !important;
        }
    </style>
    """
    
    html_table = table_summary.to_html(index=False) if not table_summary.empty else "<p>Sem dados.</p>"
    
    container = f"""
    {table_style}
    <div class="custom-table-wrapper">
        {html_table}
    </div>
    """
    
    display(HTML(container))


Problema,Objetivos,Avaliações,Algoritmo,Hypervolume,Chamadas F.O.,Avaliações DVL,Avaliações MOEA,Treino Modelos (s),Avaliação F.O. (s),Tempo Total (s),Sucesso
DTLZ1,3,250,MOEA/D,0.000000 ± 0.000000,250.0 ± 0.0,0,250,0.000000 ± 0.000000,0.006766 ± 0.000413,0.047036 ± 0.004411,20/20
DTLZ1,3,250,MOEA/D + DVL 25%,0.000000 ± 0.000000,250.0 ± 0.0,62,188,0.182744 ± 0.020287,0.006737 ± 0.001069,0.262943 ± 0.030635,20/20
DTLZ1,3,250,MOEA/D + DVL 50%,0.041101 ± 0.068800,250.0 ± 0.0,125,125,0.428049 ± 0.031572,0.006378 ± 0.000427,0.545506 ± 0.041090,20/20
DTLZ1,3,250,MOEA/D + DVL 75%,0.297584 ± 0.103892,278.0 ± 0.0,188,90,0.757339 ± 0.027540,0.006840 ± 0.000273,0.916440 ± 0.033168,20/20
DTLZ1,3,500,MOEA/D,0.000000 ± 0.000000,500.0 ± 0.0,0,500,0.000000 ± 0.000000,0.013523 ± 0.000127,0.096032 ± 0.007743,20/20
DTLZ1,3,500,MOEA/D + DVL 25%,0.043276 ± 0.078597,500.0 ± 0.0,125,375,0.408919 ± 0.004705,0.013198 ± 0.000381,0.570785 ± 0.007790,20/20
DTLZ1,3,500,MOEA/D + DVL 50%,0.455387 ± 0.158110,500.0 ± 0.0,250,250,1.121221 ± 0.014312,0.012794 ± 0.000460,1.311291 ± 0.015767,20/20
DTLZ1,3,500,MOEA/D + DVL 75%,0.561869 ± 0.004730,500.0 ± 0.0,375,125,2.382670 ± 0.014517,0.012602 ± 0.001259,2.556250 ± 0.015917,20/20
DTLZ1,3,1000,MOEA/D,0.000000 ± 0.000000,1000.0 ± 0.0,0,1000,0.000000 ± 0.000000,0.027373 ± 0.000315,0.193817 ± 0.002496,20/20
DTLZ1,3,1000,MOEA/D + DVL 25%,0.303340 ± 0.281705,1000.0 ± 0.0,250,750,1.124769 ± 0.013171,0.026543 ± 0.000316,1.410408 ± 0.015025,20/20


Problema,Objetivos,Avaliações,Algoritmo,Hypervolume,Chamadas F.O.,Avaliações DVL,Avaliações MOEA,Treino Modelos (s),Avaliação F.O. (s),Tempo Total (s),Sucesso
DTLZ1,3,250,NSGA-II,0.000000 ± 0.000000,250.0 ± 0.0,0,250,0.000000 ± 0.000000,0.006542 ± 0.000228,0.060726 ± 0.001585,20/20
DTLZ1,3,250,NSGA-II + DVL 25%,0.000000 ± 0.000000,250.0 ± 0.0,62,188,0.177343 ± 0.003688,0.006372 ± 0.000090,0.254369 ± 0.004732,20/20
DTLZ1,3,250,NSGA-II + DVL 50%,0.052135 ± 0.077507,250.0 ± 0.0,125,125,0.415949 ± 0.012137,0.006359 ± 0.000387,0.524137 ± 0.016146,20/20
DTLZ1,3,250,NSGA-II + DVL 75%,0.297584 ± 0.103892,250.0 ± 0.0,188,62,0.753657 ± 0.021763,0.006245 ± 0.000126,0.904317 ± 0.025971,20/20
DTLZ1,3,500,NSGA-II,0.000000 ± 0.000000,500.0 ± 0.0,0,500,0.000000 ± 0.000000,0.013056 ± 0.000198,0.161781 ± 0.010769,20/20
DTLZ1,3,500,NSGA-II + DVL 25%,0.106783 ± 0.082629,500.0 ± 0.0,125,375,0.410960 ± 0.006320,0.012777 ± 0.000192,0.606251 ± 0.008431,20/20
DTLZ1,3,500,NSGA-II + DVL 50%,0.564748 ± 0.070927,500.0 ± 0.0,250,250,1.121199 ± 0.012473,0.012741 ± 0.000329,1.329193 ± 0.015006,20/20
DTLZ1,3,500,NSGA-II + DVL 75%,0.567710 ± 0.019912,500.0 ± 0.0,375,125,2.392604 ± 0.015024,0.012367 ± 0.000127,2.560021 ± 0.015263,20/20
DTLZ1,3,1000,NSGA-II,0.000000 ± 0.000000,1000.0 ± 0.0,0,1000,0.000000 ± 0.000000,0.026258 ± 0.000315,0.356820 ± 0.004243,20/20
DTLZ1,3,1000,NSGA-II + DVL 25%,0.617975 ± 0.066854,1000.0 ± 0.0,250,750,1.127590 ± 0.013296,0.025478 ± 0.000210,1.519259 ± 0.022884,20/20


Problema,Objetivos,Avaliações,Algoritmo,Hypervolume,Chamadas F.O.,Avaliações DVL,Avaliações MOEA,Treino Modelos (s),Avaliação F.O. (s),Tempo Total (s),Sucesso
DTLZ1,3,250,NSGA-III,0.525638 ± 0.124308,250.0 ± 0.0,0,250,0.000000 ± 0.000000,0.006405 ± 0.000105,0.172948 ± 0.008688,20/20
DTLZ1,3,250,NSGA-III + DVL 25%,0.000000 ± 0.000000,250.0 ± 0.0,62,188,0.175353 ± 0.001959,0.006352 ± 0.000161,0.273146 ± 0.020917,20/20
DTLZ1,3,250,NSGA-III + DVL 50%,0.074570 ± 0.087685,250.0 ± 0.0,125,125,0.410525 ± 0.007387,0.006247 ± 0.000094,0.558562 ± 0.016267,20/20
DTLZ1,3,250,NSGA-III + DVL 75%,0.341454 ± 0.108644,250.0 ± 0.0,188,62,0.736847 ± 0.004229,0.006220 ± 0.000124,0.907183 ± 0.005199,20/20
DTLZ1,3,500,NSGA-III,0.536662 ± 0.114459,500.0 ± 0.0,0,500,0.000000 ± 0.000000,0.013290 ± 0.000434,0.418341 ± 0.021188,20/20
DTLZ1,3,500,NSGA-III + DVL 25%,0.103527 ± 0.084194,500.0 ± 0.0,125,375,0.410926 ± 0.005856,0.012853 ± 0.000261,0.657865 ± 0.033592,20/20
DTLZ1,3,500,NSGA-III + DVL 50%,0.568600 ± 0.069659,500.0 ± 0.0,250,250,1.127350 ± 0.016161,0.012621 ± 0.000415,1.365760 ± 0.036278,20/20
DTLZ1,3,500,NSGA-III + DVL 75%,0.611492 ± 0.018208,500.0 ± 0.0,375,125,2.397888 ± 0.019627,0.012348 ± 0.000116,2.605985 ± 0.020067,20/20
DTLZ1,3,1000,NSGA-III,0.574770 ± 0.100498,1000.0 ± 0.0,0,1000,0.000000 ± 0.000000,0.026602 ± 0.000318,0.841897 ± 0.043057,20/20
DTLZ1,3,1000,NSGA-III + DVL 25%,0.646804 ± 0.085189,1000.0 ± 0.0,250,750,1.126166 ± 0.012817,0.025585 ± 0.000210,1.688243 ± 0.066941,20/20
